# Qwen 27B Uncensored — Free Colab + Cloudflare Tunnel
### Free T4 GPU · No credit card · No port forwarding

**Step 1:** Runtime > Change runtime type > **T4 GPU** > Save
**Step 2:** Runtime > **Run all**
**Step 3:** Copy `SELF_HOSTED_BASE_URL=...` from the last cell into your JARVIS `.env`

**Keep this Colab tab open** while using JARVIS.
If it disconnects, re-run **Cells 4 and 5 only**.

In [ ]:
# ============================================
# CELL 1: Check GPU + Install dependencies
# ============================================

!nvidia-smi

# Install system packages
!apt-get update -qq && apt-get install -y -qq git-lfs wget curl build-essential cmake git > /dev/null 2>&1

# Install huggingface_hub for downloading the model
!pip install -q huggingface-hub

# Download cloudflared (x86_64 binary - Colab runs x86_64 Linux)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Verify cloudflared works
import subprocess
r = subprocess.run(['/usr/local/bin/cloudflared', '--version'], capture_output=True, text=True)
print(f'cloudflared: {r.stdout.strip()}')

# Check VRAM
result = subprocess.run(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'], capture_output=True, text=True)
vram_mb = int(result.stdout.strip().split('.')[0])
vram_gb = vram_mb / 1024
print(f'VRAM: {vram_gb:.1f} GB')
if vram_gb < 14:
    raise RuntimeError(f'Need 14GB+ VRAM but got {vram_gb:.1f}GB. Change runtime type to T4 GPU!')
print('Cell 1 DONE - GPU OK, all deps installed')

In [ ]:
# ============================================
# CELL 2: Download pre-built GGUF from HuggingFace
# ============================================
# Repo:   JonathanColetti/Qwen3.8-27B-Uncensored-GGUF  (VERIFIED public)
# File:   Qwen3.8-27B-Uncensored-noMTP-IQ2_M.gguf   (VERIFIED 10.17 GB)
# Why IQ2_M: ~10GB VRAM, fits T4 16GB with room for 4096-token context buffer

import os

GGUF_DIR = '/content/model'
GGUF_FILE = os.path.join(GGUF_DIR, 'Qwen3.8-27B-Uncensored-noMTP-IQ2_M.gguf')
HF_REPO = 'JonathanColetti/Qwen3.8-27B-Uncensored-GGUF'
HF_FILE = 'Qwen3.8-27B-Uncensored-noMTP-IQ2_M.gguf'

if os.path.exists(GGUF_FILE) and os.path.getsize(GGUF_FILE) > 1e9:
    print(f'GGUF already exists: {os.path.getsize(GGUF_FILE)/1e9:.2f} GB')
else:
    os.makedirs(GGUF_DIR, exist_ok=True)
    print(f'Downloading {HF_FILE} ...')
    print('This is ~10 GB - may take 5-15 min depending on Colab bandwidth.')
    from huggingface_hub import hf_hub_download
    path = hf_hub_download(
        repo_id=HF_REPO,
        filename=HF_FILE,
        local_dir=GGUF_DIR,
        local_dir_use_symlinks=False
    )
    actual = os.path.getsize(path)
    print(f'Downloaded: {actual/1e9:.2f} GB')
    if actual < 5e9:
        raise RuntimeError(f'Download too small ({actual/1e9:.2f} GB) - file may be corrupted')

print(f'Model ready: {GGUF_FILE}')

In [ ]:
# ============================================
# CELL 3: Build llama.cpp server with CUDA
# ============================================

import os

LLAMA_DIR = '/content/llama.cpp'
BUILD_DIR = os.path.join(LLAMA_DIR, 'build')
SERVER_BIN = os.path.join(BUILD_DIR, 'bin', 'llama-server')

if os.path.exists(SERVER_BIN):
    print('llama.cpp already built, skipping')
else:
    if not os.path.exists(LLAMA_DIR):
        print('Cloning llama.cpp...')
        !git clone https://github.com/ggml-org/llama.cpp.git $LLAMA_DIR

    # Build with CUDA support
    # GGML_CUDA=ON  : enable GPU acceleration (required for T4)
    # LLAMA_CURL=ON : enable HTTP model loading
    print('Building llama.cpp with CUDA (takes 5-8 min)...')
    !cmake -S $LLAMA_DIR -B $BUILD_DIR -DGGML_CUDA=ON -DLLAMA_CURL=ON -DCMAKE_BUILD_TYPE=Release > /content/cmake_log.txt 2>&1
    !cmake --build $BUILD_DIR -j$(nproc) --target llama-server > /content/build_log.txt 2>&1

    if not os.path.exists(SERVER_BIN):
        print('BUILD FAILED! Last 30 lines of build log:')
        !tail -30 /content/build_log.txt
        raise RuntimeError('llama.cpp build failed - check log above')

    print('llama.cpp built successfully')

print(f'Server binary: {SERVER_BIN}')

In [ ]:
# ============================================
# CELL 4: Start llama.cpp server
# ============================================
# Flags:
#   -m           : GGUF model file
#   -c 4096      : context window (4096 tokens, fits T4 VRAM)
#   -ngl 99      : offload all layers to GPU
#   --host 0.0.0.0 : listen all interfaces (needed for tunnel)
#   --port 8080  : server port
#   --parallel 2 : 2 concurrent requests
#   -ctk q4_0    : quantize KV cache K to q4_0 (saves ~1GB VRAM)
#   -ctv q4_0    : quantize KV cache V to q4_0 (saves ~1GB VRAM)
#   -t 2         : CPU threads (GPU does the work)

import subprocess, time, urllib.request, os

PORT = 8080

# Force-kill anything on port 8080 (handles stale processes from re-runs)
subprocess.run(['fuser', '-k', f'{PORT}/tcp'], capture_output=True)
subprocess.run(['pkill', '-9', '-f', 'llama-server'], capture_output=True)
time.sleep(2)

# Verify model file
if not os.path.exists(GGUF_FILE):
    raise RuntimeError(f'Model not found: {GGUF_FILE} - re-run Cell 2')

print(f'Starting server with: {GGUF_FILE}')
print(f'Model size: {os.path.getsize(GGUF_FILE)/1e9:.2f} GB')

server_proc = subprocess.Popen(
    [SERVER_BIN,
     '-m', GGUF_FILE,
     '-c', '4096',
     '-ngl', '99',
     '--host', '0.0.0.0',
     '--port', str(PORT),
     '--parallel', '2',
     '-ctk', 'q4_0', '-ctv', 'q4_0',
     '-t', '2'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

print(f'Server PID={server_proc.pid} on port {PORT}')
print('Loading model into VRAM (1-3 min on T4)...')

# Check for early crash after 15s
time.sleep(15)
if server_proc.poll() is not None:
    print('SERVER CRASHED during startup!')
    output = server_proc.stdout.read().decode(errors='replace')
    print(output[-3000:])
    raise RuntimeError('Server crashed - see error above')

# Poll /v1/models until ready
print('Server alive, waiting for model to fully load...')
ready = False
for i in range(180):
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/v1/models', timeout=2)
        print(f'Ready! Model loaded in ~{(i+1)*2}s')
        ready = True
        break
    except Exception:
        if server_proc.poll() is not None:
            print('SERVER CRASHED while loading!')
            output = server_proc.stdout.read().decode(errors='replace')
            print(output[-3000:])
            raise RuntimeError('Server crashed during loading - see error above')
        time.sleep(2)
        if (i + 1) % 15 == 0:
            print(f'  still loading... ({(i+1)*2}s)')

if not ready:
    raise RuntimeError('Server did not become ready after 6 min')

print('Cell 4 DONE - server is running')

In [ ]:
# ============================================
# CELL 5: Cloudflare Tunnel + Get URL + Test
# ============================================

import re, threading, urllib.request, json, subprocess, time

tunnel_url = [None]

# Kill existing tunnel
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(1)

# Start Cloudflare quick tunnel (free, no account needed)
tunnel_proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

def watch_tunnel():
    for raw in tunnel_proc.stdout:
        line = raw.decode(errors='replace')
        m = re.search(r'https://([a-z0-9-]+\.trycloudflare\.com)', line)
        if m:
            tunnel_url[0] = m.group(1)
            print()
            print('=' * 55)
            print('TUNNEL: https://' + tunnel_url[0])
            print('=' * 55)
            break

threading.Thread(target=watch_tunnel, daemon=True).start()
print('Waiting for Cloudflare tunnel URL (usually 5-15s)...')

for i in range(60):
    if tunnel_url[0]:
        break
    time.sleep(1)
    if (i + 1) % 10 == 0:
        print(f'  waiting for tunnel... ({i+1}s)')

if tunnel_url[0]:
    BASE = 'https://' + tunnel_url[0]
else:
    BASE = f'http://localhost:{PORT}'
    print()
    print('WARNING: Tunnel URL not found after 60s.')
    print(f'Falling back to local: {BASE}')

print()
print('*' * 55)
print('COPY THIS INTO YOUR JARVIS .env FILE:')
print(f'SELF_HOSTED_BASE_URL={BASE}')
print('*' * 55)

# Test with a real chat completion request
print()
print('Testing model response...')
try:
    payload = json.dumps({
        'model': 'qwen',
        'messages': [{'role': 'user', 'content': 'Say hello in exactly 5 words.'}],
        'max_tokens': 50,
        'stream': False
    }).encode()
    req = urllib.request.Request(
        BASE + '/v1/chat/completions',
        data=payload,
        headers={'Content-Type': 'application/json'}
    )
    resp = urllib.request.urlopen(req, timeout=120)
    data = json.loads(resp.read())
    msg = data['choices'][0]['message']['content']
    print(f'Model response: {msg}')
    print()
    print('ALL WORKING! Keep this Colab open while using JARVIS.')
except Exception as e:
    print(f'Test error: {e}')
    print('If model still loading, wait 1-2 min and re-run this cell.')

## Done!

**Keep this Colab tab open.** If it disconnects, re-run **Cells 4 and 5 only**.

Quick tunnel URLs change every time you reconnect.
For a permanent URL, sign up at https://dash.cloudflare.com/ (still free).